In [1]:
import os

import rasterio
import pandas as pd
import geopandas as gpd
import numpy as np

In [2]:
folder = '/home/jan/Documents/EPFL/project_2/data/tmp'

In [3]:
asia = gpd.read_file(os.path.join(folder, 'asia_grid_5.gpkg'))
africa = gpd.read_file(os.path.join(folder, 'africa_grid_5.gpkg'))
america = gpd.read_file(os.path.join(folder, 'america_grid_5.gpkg'))

In [4]:
grid = gpd.GeoDataFrame(pd.concat([asia,africa,america]))

In [7]:
labels = pd.read_csv('../data/labels/dataset_v5/labels_no_regrowth.csv')
labels['geometry'] = gpd.GeoSeries.from_wkt(labels['geometry'])
labels = gpd.GeoDataFrame(labels)

In [8]:
def intersect_using_spatial_index(source_gdf, intersecting_gdf):
    """
    Conduct spatial intersection using spatial index for candidates GeoDataFrame to make queries faster.
    Note, with this function, you can have multiple Polygons in the 'intersecting_gdf' and it will return all the points 
    intersect with ANY of those geometries.
    """
    source_sindex = source_gdf.sindex
    possible_matches_index = []
    
    # 'itertuples()' function is a faster version of 'iterrows()'
    for other in intersecting_gdf.itertuples():
        bounds = other.geometry.bounds
        c = list(source_sindex.intersection(bounds))
        possible_matches_index += c
    
    # Get unique candidates
    unique_candidate_matches = list(set(possible_matches_index))
    possible_matches = source_gdf.iloc[unique_candidate_matches]

    # Conduct the actual intersect
    result = possible_matches.loc[possible_matches.intersects(intersecting_gdf.unary_union)]
    return result

In [ ]:
grid
intersect_using_spatial_index()

In [13]:
grid.crs = 4326
labels.crs = 4326

In [14]:
grid.intersects(labels)

/tmp/ipykernel_1048145/1771210823.py:1: UserWarning: The indices of the two GeoSeries are different.
  grid.intersects(labels)


0        False
0        False
0        False
1        False
1        False
         ...  
30366    False
30367    False
30368    False
30369    False
30370    False
Length: 30755, dtype: bool

In [17]:
intersecting_polygons = grid[grid.intersects(labels.unary_union)]


In [20]:
intersecting_polygons

,geometry
38,"POLYGON ((167.50000 -25.00000, 167.50000 -20.0..."
58,"POLYGON ((167.50000 -20.00000, 167.50000 -15.0..."
59,"POLYGON ((172.50000 -20.00000, 172.50000 -15.0..."
69,"POLYGON ((122.50000 -15.00000, 122.50000 -10.0..."
70,"POLYGON ((127.50000 -15.00000, 127.50000 -10.0..."
...,...
184,"POLYGON ((-67.00000 17.00000, -67.00000 22.000..."
185,"POLYGON ((-62.00000 17.00000, -62.00000 22.000..."
194,"POLYGON ((-97.00000 22.00000, -97.00000 27.000..."
197,"POLYGON ((-82.00000 22.00000, -82.00000 27.000..."


In [36]:
splits = []
for idx, cell in intersecting_polygons.iterrows():
    pts = labels.loc[labels.geometry.intersects(cell.geometry)]
    split = pts.split.unique()
    assert len(split) == 1
    splits.append(split[0])


In [37]:
intersecting_polygons['split'] = splits

/home/jan/miniconda3/envs/torch/lib/python3.11/site-packages/geopandas/geodataframe.py:1543: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [39]:
intersecting_polygons.to_file('../data/tmp/grid.gpkg')